# Customer Feedback Sentiment Analysis using RNN/LSTM

**Pipeline:** Raw text → Preprocessing → Tokenisation → Embedding → LSTM → Sentiment Class

**Dataset:** Amazon Product Reviews  
**Task:** 3-class classification — Positive / Negative / Neutral  
**Architecture:** Embedding + Bidirectional LSTM + Dense  


## 1. Install & Import Dependencies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, Bidirectional, LSTM, Dense, Dropout, GlobalMaxPooling1D
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')


## 2. Load Dataset

In [ ]:
# ── Option A: Load from Kaggle (Amazon Reviews) ───────────────────────────
# df = pd.read_csv('amazon_reviews.csv')

# ── Option B: Synthetic demo dataset (runs without downloading) ───────────
np.random.seed(42)
positive_reviews = [
    'This product is absolutely amazing and works perfectly',
    'Great quality, fast shipping, very happy with purchase',
    'Excellent product highly recommend to everyone',
    'Best purchase I have made this year love it',
    'Outstanding quality exceeded all my expectations',
    'Very satisfied with this product great value for money',
    'Fantastic product works exactly as described',
    'Perfect exactly what I needed highly recommended',
    'Wonderful product will definitely buy again',
    'Love this product it is exactly what I was looking for',
]
negative_reviews = [
    'Terrible product broke after one day complete waste of money',
    'Worst purchase ever quality is extremely poor',
    'Very disappointed product does not work as advertised',
    'Awful experience product arrived damaged and defective',
    'Complete garbage would not recommend to anyone',
    'Poor quality stopped working within a week',
    'Horrible product waste of money do not buy',
    'Very bad quality fell apart immediately after use',
    'Disgusting quality product is nothing like description',
    'Terrible waste of time and money absolute disappointment',
]
neutral_reviews = [
    'Product is okay nothing special just average quality',
    'Decent product does what it says nothing more',
    'Average quality for the price not great not terrible',
    'It is acceptable but there are better options available',
    'Mediocre product does the job but nothing impressive',
    'Product is fine meets basic requirements nothing extra',
    'Ok product works as expected but quality could be better',
    'Not bad not great just a regular average product',
    'Satisfactory product nothing outstanding about it',
    'Product works but I expected better for the price',
]

# Expand to ~3000 samples for demo
reviews = (positive_reviews * 100 + negative_reviews * 100 + neutral_reviews * 100)
labels  = (['positive'] * 1000 + ['negative'] * 1000 + ['neutral'] * 1000)

df = pd.DataFrame({'review': reviews, 'sentiment': labels})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
print(df['sentiment'].value_counts())
df.head()


## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
counts = df['sentiment'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#2ecc71','#e74c3c','#3498db'])
axes[0].set_title('Sentiment Class Distribution')
axes[0].set_ylabel('Count')

# Review length distribution
df['length'] = df['review'].apply(lambda x: len(x.split()))
axes[1].hist(df['length'], bins=30, color='steelblue', edgecolor='white')
axes[1].set_title('Review Length Distribution (words)')
axes[1].set_xlabel('Word Count')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Average review length: {df["length"].mean():.1f} words')


## 4. Text Preprocessing

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text: str) -> str:
    """Lowercase, remove punctuation, stopwords, and lemmatize."""
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)            # remove HTML tags
    text = re.sub(r'http\S+|www\.\S+', '', text) # remove URLs
    text = re.sub(r'[^a-z\s]', '', text)         # remove non-alpha
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
    return ' '.join(tokens)

df['clean_review'] = df['review'].apply(preprocess_text)

print('Before:', df['review'].iloc[0])
print('After: ', df['clean_review'].iloc[0])


## 5. Tokenisation & Sequence Padding

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────
VOCAB_SIZE   = 10000
MAX_LEN      = 100
EMBED_DIM    = 64
LSTM_UNITS   = 64
BATCH_SIZE   = 32
EPOCHS       = 20
NUM_CLASSES  = 3

# ── Label encoding ────────────────────────────────────────────────────────
le = LabelEncoder()
y  = le.fit_transform(df['sentiment'])   # negative=0, neutral=1, positive=2
y_cat = to_categorical(y, num_classes=NUM_CLASSES)
print('Classes:', le.classes_)

# ── Train/val/test split ──────────────────────────────────────────────────
X_train, X_tmp, y_train, y_tmp = train_test_split(
    df['clean_review'].values, y_cat, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, random_state=42
)
print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

# ── Tokenise and pad ──────────────────────────────────────────────────────
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN, padding='post')
X_val_seq   = pad_sequences(tokenizer.texts_to_sequences(X_val),   maxlen=MAX_LEN, padding='post')
X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(X_test),  maxlen=MAX_LEN, padding='post')

print(f'Vocabulary size: {len(tokenizer.word_index)}')
print(f'Padded sequence shape: {X_train_seq.shape}')


## 6. Model Architecture — Bidirectional LSTM

In [ ]:
def build_model(vocab_size, embed_dim, max_len, lstm_units, num_classes):
    model = Sequential([
        Embedding(vocab_size, embed_dim, input_length=max_len),
        Bidirectional(LSTM(lstm_units, return_sequences=True)),
        Dropout(0.3),
        Bidirectional(LSTM(lstm_units // 2)),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(num_classes, activation='softmax')
    ], name='BiLSTM_Sentiment')
    return model

model = build_model(VOCAB_SIZE, EMBED_DIM, MAX_LEN, LSTM_UNITS, NUM_CLASSES)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()


## 7. Training

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

history = model.fit(
    X_train_seq, y_train,
    validation_data=(X_val_seq, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)


## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'],     label='Train Acc')
axes[0].plot(history.history['val_accuracy'], label='Val Acc')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['loss'],     label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Evaluation

In [ ]:
y_pred_prob = model.predict(X_test_seq)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

print(f'Test Accuracy: {accuracy_score(y_true, y_pred)*100:.2f}%\n')
print('Classification Report:')
print(classification_report(y_true, y_pred, target_names=le.classes_))


In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.title('Confusion Matrix — BiLSTM Sentiment Classifier')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Inference — Predict on New Text

In [ ]:
def predict_sentiment(text: str) -> str:
    clean = preprocess_text(text)
    seq   = pad_sequences(tokenizer.texts_to_sequences([clean]), maxlen=MAX_LEN, padding='post')
    prob  = model.predict(seq, verbose=0)[0]
    label = le.classes_[np.argmax(prob)]
    confidence = prob.max() * 100
    return f'{label.upper()} ({confidence:.1f}% confidence)'

test_reviews = [
    'This product is absolutely fantastic, highly recommended!',
    'Terrible quality, broke after one day. Complete waste of money.',
    'It is an average product, does the job but nothing special.',
]
for review in test_reviews:
    print(f'Review  : {review}')
    print(f'Predicted: {predict_sentiment(review)}\n')


## 11. Save Model & Tokenizer

In [ ]:
import pickle
model.save('sentiment_lstm_model.h5')
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
print('Model and tokenizer saved.')
